## Group-level RSA (CWS vs. CWNS) on GLMsingle single-trial ROI RDMs

Compares CWS and CWNS on the per-subject, per-ROI crossnobis RDMs computed by `GLMsingle_rsa-roi.py`, run **separately for each of the 5 noise levels** (`Q, 8, 0, n2, n6`) rather than pooling all 80 conditions into one RDM -- mirrors `univariate_fmri/group_level_all_ROI.ipynb`'s per-SNR-level structure. Four analyses:
1. **Model-fit comparison** (primary, per noise level): correlate each subject's empirical RDM (per ROI) against model RDMs -- categorical (syllable identity, speaker identity; the SNR model is degenerate and skipped at every single-noise-level tag) and, for `noiselevel-Q` only, acoustic (Praat-derived speech-metric features from `FMRI_STIM_Q_speech_metrics.xlsx`) -- then compare model-fit strength between groups via subject-level bootstrap.
2. **Direct RDM comparison** (descriptive, per noise level): compare CWS-mean vs. CWNS-mean RDM per ROI directly, no categorical models involved.
3. **Compare across noise levels** (new): combine the `syllable`/`speaker` model-fit scalars across all 5 noise levels (the only two models common to every level) and test for a linear noise-level trend, the same [2, 1, 0, -1, -2] (cleanest-to-noisiest) contrast used in `group_level_all_ROI.ipynb`.
4. **Noise ceiling** (per ROI, per group, per noise level): Nili et al. 2014 upper/lower bounds.

**Multiple comparisons note (deliberate choice):** every ROI x model test is independent, uncorrected across ROIs/models *within* a given noise level's table -- consistent with the same choice made in `univariate_group-level.ipynb`. The noise-level-trend tests are FDR-corrected separately, across their own ROI x model family.

**Data availability:** noise levels without `GLMsingle_rsa-roi.py` output yet are skipped automatically (with a printed message), not treated as an error -- run `loop_run_GLMsingle_rsa-roi.sh` (submits one job per subject x noise_level) or `sbatch run_GLMsingle_rsa-roi.sh <sub_label> <noise_level>` for a single subject/level to fill in the rest.

In [ ]:
import os
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import rsatoolbox
from scipy.spatial.distance import pdist, squareform
from statsmodels.stats.multitest import multipletests


### Set parameters

In [ ]:
bidsroot = os.path.join('/bgfs/bchandrasekaran/krs228/data/',
                        'SSP/',
                        'data_bids')
glmsingle_dir = os.path.join(bidsroot, 'derivatives', 'glmsingle')

RDM_METHOD = 'crossnobis'
# Every noise level the badaga task's SNR manipulation includes, cleanest to noisiest -- matches
# univariate_fmri/group_level_all_ROI.ipynb's contrast_list convention exactly (same order, same
# labels) so results from the two pipelines can be compared directly.
NOISE_LEVEL_TAGS = ['Q', '8', '0', 'n2', 'n6']

# Acoustic model RDMs (below) only exist for noiselevel-Q -- the only noise level the speech
# metrics sheet has measurements for (noise is added digitally on top of the same clean
# recording, so acoustic properties are only ever measured once per syllable/speaker). Skipped
# automatically (via build_acoustic_model_rdms' own guard) for every other tag.
ACOUSTIC_METRICS_XLSX = os.path.join('..', 'FMRI_STIM_Q_speech_metrics.xlsx')

rdm_dir = os.path.join(glmsingle_dir, f'rsa-roi_glmsingle_rdmcalc-{RDM_METHOD}')
print('rdm_dir:', rdm_dir)

# Ordered all-L-then-all-R (mirrored, same relative region order in both halves) rather than
# per-network L/R blocks -- kept in sync with GLMsingle_mask-betas.py / GLMsingle_rsa-roi.py.
CORTICAL_ROI_LIST = [
    'L-HG', 'L-PT', 'L-PP', 'L-STGp', 'L-STGa', 'L-ParsOp', 'L-ParsTri', 'L-SMGa', 'L-SMGp', 'L-Ang',
    'R-HG', 'R-PT', 'R-PP', 'R-STGp', 'R-STGa', 'R-ParsOp', 'R-ParsTri', 'R-SMGa', 'R-SMGp', 'R-Ang',
]

# base region order (no hemisphere prefix), for the hemisphere-hue group-specific plots below
BASE_REGION_ORDER = [roi.split('-', 1)[1] for roi in CORTICAL_ROI_LIST if roi.startswith('L-')]

N_BOOT = 10000
RNG = np.random.default_rng(0)

out_dir = os.path.join(glmsingle_dir, 'rsa-group_glmsingle')
os.makedirs(out_dir, exist_ok=True)


### Load participants and group membership

In [ ]:
participants_fpath = os.path.join(bidsroot, 'participants.tsv')
participants_df = pd.read_csv(participants_fpath, sep='\t')

# case-/whitespace-normalized group comparison -- same fix already applied in
# univariate_group-level.ipynb, since participants.tsv's group column has been observed with
# inconsistent casing (e.g. 'CWS' vs 'cws'). Noise-level-specific RDM availability is checked
# separately per tag below (run_rsa_for_noise_level), not here -- this is just the full roster.
group_norm = participants_df.group.str.strip().str.lower()
sub_list_cwns_all = list(participants_df.participant_id[group_norm == 'control'])
sub_list_cws_all = list(participants_df.participant_id[group_norm == 'cws'])

print(f'{len(sub_list_cwns_all)} CWNS, {len(sub_list_cws_all)} CWS in participants.tsv')


### Shared functions

Model building, statistics, and plotting helpers -- defined once, reused for every noise level
below. None of these depend on which noise level is currently being processed (that's always
passed in explicitly), so they don't need to be redefined per noise level.

In [ ]:
def load_subject_rdms(sub_id, noise_level_tag):
    fpath = os.path.join(rdm_dir, f'{sub_id}_glmsingle_cortical_{RDM_METHOD}_noiselevel-{noise_level_tag}_rdms.hdf5')
    # NOTE: rsatoolbox's exact load function/path is not independently confirmed against an
    # installed version (none available locally) -- rsatoolbox.rdm.rdms.load_rdm is the
    # symmetric counterpart to rsatoolbox.rdm.rdms.concat, which GLMsingle_rsa-roi.py already
    # uses successfully to save these files. Confirm this exact call on first real run; if it
    # differs, this is the one line to fix.
    return rsatoolbox.rdm.rdms.load_rdm(fpath, file_type='hdf5')


In [ ]:
def build_categorical_model_rdms(pattern_labels):
    """Build SNR/syllable/speaker categorical model RDMs (0 if the two stimuli match on that
    dimension, 1 if they differ), sized and ORDERED to exactly match `pattern_labels` -- the
    actual condition order taken from a loaded empirical RDM's pattern_descriptors, not a
    freshly re-derived list. The model and empirical RDMs' row/column order must match exactly
    for rsatoolbox.rdm.compare() to compare like-for-like conditions.

    Each condition label is expected to be 'SYLLABLE_SPEAKER_NOISELEVEL' (e.g. 'BA_F1_Q'), per
    GLMsingle_first-level.py's build_condition_labels(). Fixes the copy-paste bug in the old
    group_level_rsa_searchlight_WIP.ipynb, where the talker/speaker model RDM was assigned into
    the syllable_rdms variable instead of its own.
    """
    n = len(pattern_labels)
    parsed = [label.split('_') for label in pattern_labels]
    for p in parsed:
        assert len(p) == 3, f"expected 'syllable_speaker_noiselevel', got {p!r}"
    syllables = [p[0] for p in parsed]
    speakers = [p[1] for p in parsed]
    noise_levels = [p[2] for p in parsed]

    model_rdms = {}
    model_rdms['syllable'] = np.array([[0 if syllables[i] == syllables[j] else 1
                                        for j in range(n)] for i in range(n)])
    model_rdms['speaker'] = np.array([[0 if speakers[i] == speakers[j] else 1
                                       for j in range(n)] for i in range(n)])

    # SNR model is degenerate (all-zero, no variance to model) if every trial shares the same
    # noise level -- always true now, since every noise level is processed as its own
    # noise-level-restricted RDM (never all 80 conditions pooled together). Skip it automatically
    # rather than including a meaningless all-zero model.
    if len(set(noise_levels)) > 1:
        model_rdms['snr'] = np.array([[0 if noise_levels[i] == noise_levels[j] else 1
                                       for j in range(n)] for i in range(n)])
    else:
        print(f'Only one noise level ({noise_levels[0]}) present -- skipping the degenerate SNR model.')

    return model_rdms


In [ ]:
ACOUSTIC_FEATURE_GROUPS = {
    'f0': ['MEAN_F0', 'STD_F0', 'MIN_F0', 'MAX_F0'],
    'intensity': ['MEAN_INTENSITY', 'STD_INTENSITY', 'MIN_INTENSITY', 'MAX_INTENSITY'],
    'formants': ['MEAN_F1', 'STD_F1', 'MIN_F1', 'MAX_F1', 'MEAN_F2', 'STD_F2', 'MIN_F2', 'MAX_F2'],
    # 'duration' deliberately excluded: DURATION and VOWEL_DURATION are numerically identical
    # for every stimulus (single-syllable CV recordings), so together they'd just double-weight
    # one real dimension against F0/intensity/formants without adding discriminative information.
}


def load_acoustic_features(xlsx_path):
    """Load the Praat-derived speech-metrics sheet, indexed by (syllable, speaker) -- the sheet
    only has one row per syllable x speaker (measured at noise_level 'Q', the only level
    acoustic properties were measured for; noise is added digitally on top of the same clean
    recording, so the underlying speech acoustics don't change across noise level).

    NOTE: the sheet's 'VOWEL' column is -- despite the name -- a Praat interval string like
    'DURATION'/'VOWEL_DURATION', not a vowel-identity label (confirmed: it differs from
    VOWEL_DURATION on every row). Most likely the narrower sub-window Praat used to sample
    F1/F2/F0 away from formant-transition edges. Not used as a feature here. DURATION and
    VOWEL_DURATION themselves are also not parsed out -- they're numerically identical for
    every stimulus (see ACOUSTIC_FEATURE_GROUPS above), so neither is used as a feature either.
    """
    df = pd.read_excel(xlsx_path)
    df['stimulus'] = df['FILE_NAME'].str.replace('.wav', '', regex=False)
    df[['syllable', 'speaker', 'noise_level']] = df['stimulus'].str.split('_', expand=True)
    return df.set_index(['syllable', 'speaker'])


def build_acoustic_model_rdms(pattern_labels, acoustic_df):
    """Continuous-valued model RDMs (Euclidean distance on z-scored features) from the Praat
    speech-metrics sheet. Only valid when every condition in `pattern_labels` is noise_level
    'Q' -- the only level acoustic features were measured for -- so this returns {} (with a
    printed message) otherwise, mirroring build_categorical_model_rdms' degenerate-model-skip
    convention rather than assuming acoustic properties are noise-invariant.
    """
    parsed = [label.split('_') for label in pattern_labels]
    noise_levels = {p[2] for p in parsed}
    if noise_levels != {'Q'}:
        print(f"Acoustic model RDMs require noise_level='Q'-only conditions (got {sorted(noise_levels)}) "
             "-- skipping (speech metrics were only measured at noise_level 'Q').")
        return {}

    def feature_matrix(columns):
        rows = [acoustic_df.loc[(p[0], p[1]), columns].to_numpy(dtype=float) for p in parsed]
        mat = np.vstack(rows)
        mean = mat.mean(axis=0)
        std = mat.std(axis=0, ddof=0)
        std_safe = np.where(std == 0, 1, std)  # guard constant columns against divide-by-zero
        return (mat - mean) / std_safe

    def euclidean_rdm(columns):
        z = feature_matrix(columns)
        return squareform(pdist(z, metric='euclidean'))

    model_rdms = {}
    all_columns = [c for cols in ACOUSTIC_FEATURE_GROUPS.values() for c in cols]
    model_rdms['acoustic'] = euclidean_rdm(all_columns)
    for group_name, columns in ACOUSTIC_FEATURE_GROUPS.items():
        model_rdms[f'acoustic_{group_name}'] = euclidean_rdm(columns)

    return model_rdms


In [ ]:
def get_roi_rdm_vector(rdms_obj, roi):
    """Extract the single dissimilarity vector for one ROI from a subject's full RDMs object.
    NOTE: .get_vectors() is rsatoolbox's compressed (upper-triangular) vector form -- confirm
    this exact method name on first real run; .get_matrices() is the square-matrix alternative
    if this doesn't match the installed rsatoolbox version.
    """
    roi_rdm = rdms_obj.subset('ROI', roi)
    return roi_rdm.get_vectors()[0]


In [ ]:
def compute_noise_ceiling(rdm_vectors):
    """Upper/lower noise-ceiling bounds (Nili et al. 2014) for one ROI's RDM vectors across a
    set of subjects (same group). Upper bound: each subject's vector correlated (Pearson, same
    method='corr' semantics used for model-fit) against the mean of ALL subjects' vectors
    (including their own) -- optimistic, since a subject's own data leaks into the reference
    it's compared against. Lower bound: each subject's vector correlated against the
    leave-one-out mean of everyone else -- an unbiased estimate of the best fit a genuinely
    novel model could achieve. Both averaged across subjects.
    """
    rdm_vectors = np.vstack(rdm_vectors)
    n = rdm_vectors.shape[0]
    mean_all = rdm_vectors.mean(axis=0)
    upper_corrs = [np.corrcoef(rdm_vectors[i], mean_all)[0, 1] for i in range(n)]

    lower_corrs = []
    for i in range(n):
        mean_loo = np.delete(rdm_vectors, i, axis=0).mean(axis=0)
        lower_corrs.append(np.corrcoef(rdm_vectors[i], mean_loo)[0, 1])

    return {'ceiling_upper': np.mean(upper_corrs), 'ceiling_lower': np.mean(lower_corrs)}


In [ ]:
def bootstrap_group_difference(values_a, values_b, n_boot=N_BOOT, rng=RNG):
    """Subject-level bootstrap test for a difference in means between two groups. Resamples
    subjects WITH replacement WITHIN each group (not across groups), computes the group-mean
    difference (a - b) each iteration, and derives a two-sided empirical p-value from how often
    the null-centered bootstrap distribution is at least as extreme as the observed difference.
    Operates purely on whatever scalars are passed in -- agnostic to how they were derived,
    including from GLMsingle-based RDMs (confirmed compatible per the earlier discussion: the
    bootstrap is downstream of and independent from how the single-trial patterns were estimated).
    """
    values_a = np.asarray(values_a)
    values_b = np.asarray(values_b)
    observed_diff = values_a.mean() - values_b.mean()

    boot_diffs = np.empty(n_boot)
    for i in range(n_boot):
        resampled_a = rng.choice(values_a, size=len(values_a), replace=True)
        resampled_b = rng.choice(values_b, size=len(values_b), replace=True)
        boot_diffs[i] = resampled_a.mean() - resampled_b.mean()

    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
    null_centered = boot_diffs - boot_diffs.mean()
    p_value = np.mean(np.abs(null_centered) >= np.abs(observed_diff))

    return {'observed_diff': observed_diff, 'ci_low': ci_low, 'ci_high': ci_high, 'p_value': p_value}


In [ ]:
def bootstrap_one_sample(values, null_value=0, n_boot=N_BOOT, rng=RNG):
    """Subject-level bootstrap one-sample test: is the mean of `values` significantly
    different from `null_value` (typically 0, i.e. no representational correlation with the
    model)? Resamples subjects with replacement within a single group, builds an empirical
    distribution of the mean, and derives a two-sided p-value/CI for whether the observed mean
    differs from null_value.
    """
    values = np.asarray(values)
    observed_mean = values.mean()

    boot_means = np.empty(n_boot)
    for i in range(n_boot):
        resampled = rng.choice(values, size=len(values), replace=True)
        boot_means[i] = resampled.mean()

    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
    # center the bootstrap distribution at null_value (removing its own sampling bias first) to
    # test how extreme the observed mean is under a null of "no effect"
    null_centered = boot_means - boot_means.mean() + null_value
    p_value = np.mean(np.abs(null_centered - null_value) >= np.abs(observed_mean - null_value))

    return {'observed_mean': observed_mean, 'ci_low': ci_low, 'ci_high': ci_high, 'p_value': p_value}


In [ ]:
def bootstrap_rdm_distance(rdm_vectors_a, rdm_vectors_b, n_boot=N_BOOT, rng=RNG):
    """Bootstrap CI for whether two groups' mean RDMs differ, using 1 - Pearson correlation
    between the two group-mean RDM vectors as the dissimilarity-of-dissimilarities statistic.
    Descriptive complement to the categorical-model-fit comparison above -- no categorical
    models involved, just "is the overall similarity structure different between groups."
    Doesn't have a natural null-hypothesis p-value the way a mean-difference bootstrap does;
    report the CI and whether it's close to 0 (little difference) or not.
    """
    rdm_vectors_a = np.vstack(rdm_vectors_a)
    rdm_vectors_b = np.vstack(rdm_vectors_b)

    def rdm_corr_distance(vecs_a, vecs_b):
        mean_a = vecs_a.mean(axis=0)
        mean_b = vecs_b.mean(axis=0)
        r = np.corrcoef(mean_a, mean_b)[0, 1]
        return 1 - r

    observed_distance = rdm_corr_distance(rdm_vectors_a, rdm_vectors_b)

    idx_a = np.arange(len(rdm_vectors_a))
    idx_b = np.arange(len(rdm_vectors_b))
    boot_distances = np.empty(n_boot)
    for i in range(n_boot):
        resampled_a = rdm_vectors_a[rng.choice(idx_a, size=len(idx_a), replace=True)]
        resampled_b = rdm_vectors_b[rng.choice(idx_b, size=len(idx_b), replace=True)]
        boot_distances[i] = rdm_corr_distance(resampled_a, resampled_b)

    ci_low, ci_high = np.percentile(boot_distances, [2.5, 97.5])
    return {'observed_distance': observed_distance, 'ci_low': ci_low, 'ci_high': ci_high}


In [ ]:
def _outline_only(ax):
    """Strip fill from stripplot dots and boxplot boxes, re-applying each hue group's original
    fill color as the outline/edge color first -- so the now-transparent shapes are still
    outlined in the color they're assigned, rather than whatever default edge color seaborn
    would otherwise leave them with.
    """
    for collection in ax.collections:
        facecolor = collection.get_facecolor()
        collection.set_edgecolor(facecolor)
        collection.set_facecolor('none')
    for patch in ax.patches:
        facecolor = patch.get_facecolor()
        patch.set_edgecolor(facecolor)
        patch.set_facecolor('none')


def _add_asterisk_headroom(ax, y_top):
    """Compute an asterisk y-position with headroom above the highest data point, and expand
    the axes' y-limit to guarantee that headroom doesn't get clipped or collide with the title.
    Uses a fixed fraction of the actual visible y-range, so it works correctly regardless of
    whether y_top is positive, negative, or near zero.
    """
    y_min, y_max = ax.get_ylim()
    y_range = y_max - y_min
    asterisk_y = y_top + 0.08 * y_range
    ax.set_ylim(y_min, y_max + 0.15 * y_range)
    return asterisk_y


GROUP_ORDER = ['CWS', 'CWNS']
GROUP_PALETTE = {'CWNS': '#009E73', 'CWS': '#CC79A7'}  # bluish-green / reddish-purple
GROUP_OFFSET = {'CWS': -0.2, 'CWNS': 0.2}  # mirrors HEMISPHERE_OFFSET's dodge convention below


def plot_roi_box_strip(model_fit_df, within_group_significance_df, noise_ceiling_df, model_name, roi_order):
    model_df = model_fit_df[model_fit_df.model == model_name]

    fig, ax = plt.subplots(1, 1, figsize=(0.5 * len(roi_order) + 2, 4), dpi=300)

    sns.stripplot(data=model_df, x='ROI', y='fit', hue='group', order=roi_order,
                 hue_order=GROUP_ORDER, palette=GROUP_PALETTE,
                 dodge=True, linewidth=0.5, size=3, legend=None, ax=ax, zorder=2)
    sns.boxplot(data=model_df, x='ROI', y='fit', hue='group', order=roi_order,
               hue_order=GROUP_ORDER, palette=GROUP_PALETTE,
               dodge=True, linewidth=1, fliersize=0, ax=ax, zorder=1)
    _outline_only(ax)

    # noise ceiling: shaded band per (ROI, group) at that group's dodge position, in the same
    # color as the group's boxes (low alpha) so it reads as "this box's ceiling," not a new
    # category. Silently skipped for any (ROI, group) noise_ceiling_df doesn't have a row for
    # (e.g. fewer than 3 subjects -- see compute_noise_ceiling).
    for roi in roi_order:
        for group_name in GROUP_ORDER:
            ceiling_row = noise_ceiling_df[
                (noise_ceiling_df.ROI == roi) & (noise_ceiling_df.group == group_name)
            ]
            if len(ceiling_row) == 0:
                continue
            x = roi_order.index(roi) + GROUP_OFFSET[group_name]
            ax.fill_between([x - 0.08, x + 0.08],
                            ceiling_row['ceiling_lower'].iloc[0], ceiling_row['ceiling_upper'].iloc[0],
                            color=GROUP_PALETTE[group_name], alpha=0.25, zorder=0, linewidth=0)

    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1), title='Group')
    ax.axhline(y=0, color='0.5', linestyle='--', linewidth=0.5)
    ax.set_ylabel('Model-fit (RDM correlation)')
    ax.set_title(f'model-{model_name}')

    # mark ROIs with ANY significant (FDR-corrected) within-group effect (either group)
    y_top = model_df['fit'].max()
    asterisk_y = _add_asterisk_headroom(ax, y_top)
    sig_rois = within_group_significance_df[
        (within_group_significance_df.model == model_name) &
        (within_group_significance_df.p_fdr < 0.05)
    ]['ROI'].unique()
    for roi in sig_rois:
        if roi in roi_order:
            ax.text(roi_order.index(roi), asterisk_y, '*', ha='center', va='bottom',
                    fontsize=14, fontweight='bold')

    fig.tight_layout()
    sns.despine(ax=ax)
    return fig


# Seaborn's default dodge for 2 hue categories with the default box width (0.8) centers each
# hue's sub-box at +/-0.2 from the category tick position.
HEMISPHERE_ORDER = ['L', 'R']
HEMISPHERE_OFFSET = {'L': -0.2, 'R': 0.2}
HEMISPHERE_PALETTE = {'L': '#0072B2', 'R': '#D55E00'}  # blue / vermillion


def plot_roi_box_strip_by_hemisphere(model_fit_df, within_group_significance_df, noise_ceiling_df,
                                     model_name, group_name, region_order):
    """Group-specific variant: x=region (hemisphere prefix stripped), hue=hemisphere. Fewer
    x-categories than plot_roi_box_strip's combined (x=ROI, hue=group) view -- one figure per
    (model, group) instead of one per model. Asterisks are placed over the specific L or R box
    (not the region's center), so significance is shown per-hemisphere, not pooled across both.
    """
    group_df = model_fit_df[(model_fit_df.model == model_name) & (model_fit_df.group == group_name)].copy()
    group_df['hemisphere'] = group_df['ROI'].str.split('-', n=1).str[0]
    group_df['region'] = group_df['ROI'].str.split('-', n=1).str[1]

    fig, ax = plt.subplots(1, 1, figsize=(0.5 * len(region_order) + 2, 4), dpi=300)

    sns.stripplot(data=group_df, x='region', y='fit', hue='hemisphere', order=region_order,
                 hue_order=HEMISPHERE_ORDER, palette=HEMISPHERE_PALETTE,
                 dodge=True, linewidth=0.5, size=3, legend=None, ax=ax, zorder=2)
    sns.boxplot(data=group_df, x='region', y='fit', hue='hemisphere', order=region_order,
               hue_order=HEMISPHERE_ORDER, palette=HEMISPHERE_PALETTE,
               dodge=True, linewidth=1, fliersize=0, ax=ax, zorder=1)
    _outline_only(ax)

    # noise ceiling: shaded band per (region, hemisphere) at that hemisphere's dodge position,
    # scoped to this plot's single group_name.
    for region_idx, region in enumerate(region_order):
        for hemi, offset in HEMISPHERE_OFFSET.items():
            roi_name = f'{hemi}-{region}'
            ceiling_row = noise_ceiling_df[
                (noise_ceiling_df.ROI == roi_name) & (noise_ceiling_df.group == group_name)
            ]
            if len(ceiling_row) == 0:
                continue
            x = region_idx + offset
            ax.fill_between([x - 0.08, x + 0.08],
                            ceiling_row['ceiling_lower'].iloc[0], ceiling_row['ceiling_upper'].iloc[0],
                            color=HEMISPHERE_PALETTE[hemi], alpha=0.25, zorder=0, linewidth=0)

    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1), title='Hemisphere')
    ax.axhline(y=0, color='0.5', linestyle='--', linewidth=0.5)
    ax.set_ylabel('Model-fit (RDM correlation)')
    ax.set_title(f'model-{model_name}, group-{group_name}')

    # mark the SPECIFIC L or R box (FDR-corrected) with an asterisk, per-hemisphere, using the
    # dodge offsets above
    y_top = group_df['fit'].max()
    asterisk_y = _add_asterisk_headroom(ax, y_top)
    sig_rois_set = set(within_group_significance_df[
        (within_group_significance_df.model == model_name) &
        (within_group_significance_df.group == group_name) &
        (within_group_significance_df.p_fdr < 0.05)
    ]['ROI'])
    for hemi, offset in HEMISPHERE_OFFSET.items():
        for region_idx, region in enumerate(region_order):
            if f'{hemi}-{region}' in sig_rois_set:
                ax.text(region_idx + offset, asterisk_y, '*', ha='center', va='bottom',
                        fontsize=14, fontweight='bold')

    fig.tight_layout()
    sns.despine(ax=ax)
    return fig


### Per-noise-level orchestration

`run_rsa_for_noise_level(tag)` runs the full pipeline (RDM loading, model building, model-fit
scalars, noise ceiling, group-comparison bootstrap, within-group bootstrap, direct RDM
comparison) for one noise level, returning `None` (with a printed message) if that tag's RDM
files don't exist on disk yet -- so noise levels that haven't been run through
`GLMsingle_rsa-roi.py` yet are skipped gracefully, not a hard `RuntimeError`, since it's expected
that not every noise level has been processed by the time this notebook is first run.

In [ ]:
def run_rsa_for_noise_level(noise_level_tag):
    print(f'--- noiselevel-{noise_level_tag} ---')

    rdm_fpaths = sorted(glob(os.path.join(rdm_dir, f'*_glmsingle_cortical_{RDM_METHOD}_noiselevel-{noise_level_tag}_rdms.hdf5')))
    if len(rdm_fpaths) == 0:
        print(f'No RDM files found for noiselevel-{noise_level_tag} -- skipping '
             f'(run GLMsingle_rsa-roi.py with --noise_level={noise_level_tag} to generate them).')
        return None

    subs_with_rdms = {os.path.basename(fp).split('_')[0] for fp in rdm_fpaths}
    sub_list_cwns = [s for s in sub_list_cwns_all if s in subs_with_rdms]
    sub_list_cws = [s for s in sub_list_cws_all if s in subs_with_rdms]
    all_subs = sub_list_cwns + sub_list_cws
    print(f'{len(sub_list_cwns)} CWNS, {len(sub_list_cws)} CWS with RDMs for noiselevel-{noise_level_tag}')

    if len(sub_list_cwns) < 3 or len(sub_list_cws) < 3:
        print(f'Fewer than 3 subjects in one or both groups for noiselevel-{noise_level_tag} -- '
             'skipping (noise ceiling and bootstrap statistics need at least 3 subjects per group).')
        return None

    subject_rdms = {sub_id: load_subject_rdms(sub_id, noise_level_tag) for sub_id in all_subs}

    # condition order/labels taken from an actual loaded RDM, not freshly re-derived -- must
    # match subject_rdms' actual pattern order for build_categorical_model_rdms/get_roi_rdm_vector
    # to line up correctly against the empirical RDMs.
    # pattern order is assumed identical across subjects/ROIs (same fixed condition design for
    # everyone) -- take it from the first subject's first ROI as the reference. Subset to a
    # single ROI first (rather than reading pattern_descriptors off the full concatenated,
    # multi-ROI RDMs object directly) and use the 'stimulus' key -- matches the descriptor='stimulus'
    # argument GLMsingle_rsa-roi.py's calc_rdm() call used to build these RDMs.
    example_rdms = subject_rdms[all_subs[0]]
    example_roi_rdm = example_rdms.subset('ROI', CORTICAL_ROI_LIST[0])
    pattern_labels = list(example_roi_rdm.pattern_descriptors['stimulus'])

    categorical_model_rdms = build_categorical_model_rdms(pattern_labels)
    acoustic_df = load_acoustic_features(ACOUSTIC_METRICS_XLSX)
    acoustic_model_rdms = build_acoustic_model_rdms(pattern_labels, acoustic_df)
    model_rdms = {**categorical_model_rdms, **acoustic_model_rdms}

    # per-subject, per-ROI model-fit scalars
    model_fit_rows = []
    for sub_id in all_subs:
        group_name = 'CWNS' if sub_id in sub_list_cwns else 'CWS'
        rdms_obj = subject_rdms[sub_id]
        for roi in CORTICAL_ROI_LIST:
            empirical_vector = get_roi_rdm_vector(rdms_obj, roi)
            for model_name, model_rdm in model_rdms.items():
                model_vector = squareform(model_rdm, checks=False)
                fit = np.corrcoef(empirical_vector, model_vector)[0, 1]
                model_fit_rows.append({'participant_id': sub_id, 'group': group_name, 'ROI': roi,
                                       'model': model_name, 'fit': fit})
    model_fit_df = pd.DataFrame(model_fit_rows)
    model_fit_df.to_csv(os.path.join(out_dir, f'model_fit_scalars_noiselevel-{noise_level_tag}.csv'), index=False)

    # noise ceiling, per ROI, per group
    noise_ceiling_rows = []
    for roi in CORTICAL_ROI_LIST:
        for group_name, sub_list in [('CWNS', sub_list_cwns), ('CWS', sub_list_cws)]:
            rdm_vectors = [get_roi_rdm_vector(subject_rdms[sub_id], roi) for sub_id in sub_list]
            ceiling = compute_noise_ceiling(rdm_vectors)
            noise_ceiling_rows.append({'ROI': roi, 'group': group_name, **ceiling})
    noise_ceiling_df = pd.DataFrame(noise_ceiling_rows)
    noise_ceiling_df.to_csv(os.path.join(out_dir, f'noise_ceiling_noiselevel-{noise_level_tag}.csv'), index=False)

    # group-comparison bootstrap (CWS - CWNS), per ROI, per model
    group_comparison_rows = []
    for roi in CORTICAL_ROI_LIST:
        for model_name in model_rdms:
            fit_cws = model_fit_df[(model_fit_df.ROI == roi) & (model_fit_df.model == model_name) &
                                   (model_fit_df.group == 'CWS')]['fit']
            fit_cwns = model_fit_df[(model_fit_df.ROI == roi) & (model_fit_df.model == model_name) &
                                    (model_fit_df.group == 'CWNS')]['fit']
            result = bootstrap_group_difference(fit_cws, fit_cwns)
            group_comparison_rows.append({'ROI': roi, 'model': model_name, **result})
    group_comparison_df = pd.DataFrame(group_comparison_rows)
    group_comparison_df['p_fdr'] = multipletests(group_comparison_df['p_value'], method='fdr_bh')[1]
    group_comparison_df.to_csv(os.path.join(out_dir, f'group_comparison_bootstrap_noiselevel-{noise_level_tag}.csv'), index=False)

    # within-group significance (fit != 0), per ROI, per model, per group
    within_group_rows = []
    for roi in CORTICAL_ROI_LIST:
        for model_name in model_rdms:
            for group_name in ['CWNS', 'CWS']:
                fit_values = model_fit_df[(model_fit_df.ROI == roi) & (model_fit_df.model == model_name) &
                                          (model_fit_df.group == group_name)]['fit']
                result = bootstrap_one_sample(fit_values)
                within_group_rows.append({'ROI': roi, 'model': model_name, 'group': group_name, **result})
    within_group_significance_df = pd.DataFrame(within_group_rows)
    within_group_significance_df['p_fdr'] = multipletests(within_group_significance_df['p_value'], method='fdr_bh')[1]
    within_group_significance_df.to_csv(os.path.join(out_dir, f'within_group_significance_noiselevel-{noise_level_tag}.csv'), index=False)

    # direct RDM comparison (descriptive, no categorical model involved)
    mean_rdm_rows = []
    for roi in CORTICAL_ROI_LIST:
        rdm_vectors_cwns = [get_roi_rdm_vector(subject_rdms[sub_id], roi) for sub_id in sub_list_cwns]
        rdm_vectors_cws = [get_roi_rdm_vector(subject_rdms[sub_id], roi) for sub_id in sub_list_cws]
        result = bootstrap_rdm_distance(rdm_vectors_cws, rdm_vectors_cwns)
        mean_rdm_rows.append({'ROI': roi, **result})
    mean_rdm_df = pd.DataFrame(mean_rdm_rows)
    mean_rdm_df.to_csv(os.path.join(out_dir, f'mean_rdm_comparison_noiselevel-{noise_level_tag}.csv'), index=False)

    return {
        'sub_list_cwns': sub_list_cwns, 'sub_list_cws': sub_list_cws,
        'subject_rdms': subject_rdms, 'model_rdms': model_rdms,
        'model_fit_df': model_fit_df, 'noise_ceiling_df': noise_ceiling_df,
        'group_comparison_df': group_comparison_df,
        'within_group_significance_df': within_group_significance_df,
        'mean_rdm_df': mean_rdm_df,
    }


In [ ]:
results_by_tag = {}
for tag in NOISE_LEVEL_TAGS:
    result = run_rsa_for_noise_level(tag)
    if result is not None:
        results_by_tag[tag] = result

print(f'\nProcessed {len(results_by_tag)}/{len(NOISE_LEVEL_TAGS)} noise levels: {list(results_by_tag.keys())}')


#### QC: visualize the categorical model RDMs

Shown for the first available noise level only (the categorical model RDMs' shape/labels differ
across noise levels only in which conditions are included, not in the underlying logic).

In [ ]:
if len(results_by_tag) == 0:
    print('No noise levels available yet -- skipping QC plots.')
    _qc_tag = None
else:
    _qc_tag = next(iter(results_by_tag))
    _qc_model_rdms = results_by_tag[_qc_tag]['model_rdms']

    fig, axes = plt.subplots(1, len(_qc_model_rdms), figsize=(4 * len(_qc_model_rdms), 4), dpi=150)
    axes = np.atleast_1d(axes)
    for ax, (model_name, model_rdm) in zip(axes, _qc_model_rdms.items()):
        sns.heatmap(model_rdm, ax=ax, cmap='viridis', square=True, cbar=True)
        ax.set_title(f'model-{model_name}')
    fig.suptitle(f'Model RDMs (noiselevel-{_qc_tag})')
    fig.tight_layout()


#### QC: visualize CWS-mean vs. CWNS-mean RDM for one ROI

Shown for the first available noise level only.

In [ ]:
if _qc_tag is None:
    print('No noise levels available yet -- skipping QC plot.')
else:
    _qc_result = results_by_tag[_qc_tag]
    example_roi = CORTICAL_ROI_LIST[0]

    rdm_vectors_cwns = [get_roi_rdm_vector(_qc_result['subject_rdms'][s], example_roi) for s in _qc_result['sub_list_cwns']]
    rdm_vectors_cws = [get_roi_rdm_vector(_qc_result['subject_rdms'][s], example_roi) for s in _qc_result['sub_list_cws']]
    mean_rdm_cwns = squareform(np.vstack(rdm_vectors_cwns).mean(axis=0))
    mean_rdm_cws = squareform(np.vstack(rdm_vectors_cws).mean(axis=0))

    fig, axes = plt.subplots(1, 2, figsize=(9, 4), dpi=150)
    sns.heatmap(mean_rdm_cwns, ax=axes[0], cmap='viridis', square=True)
    axes[0].set_title(f'CWNS mean RDM: {example_roi}')
    sns.heatmap(mean_rdm_cws, ax=axes[1], cmap='viridis', square=True)
    axes[1].set_title(f'CWS mean RDM: {example_roi}')
    fig.suptitle(f'noiselevel-{_qc_tag}')
    fig.tight_layout()


### Boxplot + stripplot per model, per noise level

One figure per (noise level, model) -- `plot_roi_box_strip`/`plot_roi_box_strip_by_hemisphere`
are unchanged from before (they never depended on `NOISE_LEVEL_TAG` internally), just called
once per available tag now instead of once total.

In [ ]:
for tag, result in results_by_tag.items():
    model_fit_df = result['model_fit_df']
    within_group_significance_df = result['within_group_significance_df']
    noise_ceiling_df = result['noise_ceiling_df']
    model_rdms = result['model_rdms']

    for model_name in model_rdms.keys():
        fig = plot_roi_box_strip(model_fit_df, within_group_significance_df, noise_ceiling_df, model_name, CORTICAL_ROI_LIST)
        fig.savefig(os.path.join(out_dir, f'boxplot_model-{model_name}_noiselevel-{tag}.png'))
        fig.savefig(os.path.join(out_dir, f'boxplot_model-{model_name}_noiselevel-{tag}.svg'))
        plt.close(fig)

        for group_name in ['CWS', 'CWNS']:
            fig = plot_roi_box_strip_by_hemisphere(model_fit_df, within_group_significance_df, noise_ceiling_df,
                                                   model_name, group_name, BASE_REGION_ORDER)
            fig.savefig(os.path.join(
                out_dir, f'boxplot_model-{model_name}_group-{group_name}_by-hemisphere_noiselevel-{tag}.png'))
            fig.savefig(os.path.join(
                out_dir, f'boxplot_model-{model_name}_group-{group_name}_by-hemisphere_noiselevel-{tag}.svg'))
            plt.close(fig)

print('Boxplots saved for', list(results_by_tag.keys()))


### Brain plots (surface-based), per noise level

In [ ]:
from roi_surface_plotting import plot_roi_surface_stat, build_mask_path_dict

masks_dir = os.path.join(bidsroot, 'derivatives', 'nilearn', 'masks')
space_label = 'MNI152NLin2009cAsym'

# network_name='dseg' passed directly (the actual mask-directory atlas name), rather than adding
# a new 'cortical' branch to build_mask_path_dict's network_name->mask_network_name mapping --
# avoids touching the copied reference module at all. Computed once -- doesn't depend on noise level.
mask_path_dict = build_mask_path_dict(CORTICAL_ROI_LIST, masks_dir, network_name='dseg', space_label=space_label)


In [ ]:
# Precompute the per-(noise level, model) stat dicts once (fast -- no plotting yet). Splitting
# the actual plotting (slow: surface.vol_to_surf projects every vertex on the whole hemisphere
# surface, per ROI, per call) into separate cells below by comparison type (cws/cwns/diff) gives
# natural checkpoints -- if interrupted partway through, whichever cells already ran have their
# plots saved, and it's clear which comparison type is left to (re-)run.
stat_dicts_by_tag_model = {}
for tag, result in results_by_tag.items():
    within_group_significance_df = result['within_group_significance_df']
    group_comparison_df = result['group_comparison_df']
    stat_dicts_by_tag_model[tag] = {}
    for model_name in result['model_rdms'].keys():
        cws_stat_dict = within_group_significance_df[
            (within_group_significance_df.model == model_name) &
            (within_group_significance_df.group == 'CWS')
        ].set_index('ROI')['observed_mean'].to_dict()
        cwns_stat_dict = within_group_significance_df[
            (within_group_significance_df.model == model_name) &
            (within_group_significance_df.group == 'CWNS')
        ].set_index('ROI')['observed_mean'].to_dict()
        diff_stat_dict = group_comparison_df[
            group_comparison_df.model == model_name
        ].set_index('ROI')['observed_diff'].to_dict()

        stat_dicts_by_tag_model[tag][model_name] = {'cws': cws_stat_dict, 'cwns': cwns_stat_dict, 'diff': diff_stat_dict}

print('noise levels x models ready to plot:', {tag: list(m.keys()) for tag, m in stat_dicts_by_tag_model.items()})


In [ ]:
def plot_and_save_surface(noise_level_tag, model_name, label, stat_dict):
    if len(stat_dict) == 0:
        print(f'No data for noiselevel-{noise_level_tag}, model-{model_name}, group-{label} -- skipping brain plot.')
        return
    fig = plot_roi_surface_stat(
        stat_dict, mask_path_dict,
        views=('lateral',),
        title=f'model-{model_name}, group-{label} (noiselevel-{noise_level_tag})',
    )
    fig.savefig(os.path.join(
        out_dir, f'surface_model-{model_name}_group-{label}_noiselevel-{noise_level_tag}.png'))
    fig.savefig(os.path.join(
        out_dir, f'surface_model-{model_name}_group-{label}_noiselevel-{noise_level_tag}.svg'))
    plt.close(fig)


#### CWS surface plots

In [ ]:
for tag, stat_dicts_by_model in stat_dicts_by_tag_model.items():
    for model_name, stat_dicts in stat_dicts_by_model.items():
        plot_and_save_surface(tag, model_name, 'cws', stat_dicts['cws'])


#### CWNS surface plots

In [ ]:
for tag, stat_dicts_by_model in stat_dicts_by_tag_model.items():
    for model_name, stat_dicts in stat_dicts_by_model.items():
        plot_and_save_surface(tag, model_name, 'cwns', stat_dicts['cwns'])


#### CWS − CWNS difference surface plots

In [ ]:
for tag, stat_dicts_by_model in stat_dicts_by_tag_model.items():
    for model_name, stat_dicts in stat_dicts_by_model.items():
        plot_and_save_surface(tag, model_name, 'diff', stat_dicts['diff'])


### Compare across noise levels

Combines model-fit scalars for the two models common to every noise level (`syllable`,
`speaker` -- `snr` is always degenerate within a single tag, and the `acoustic*` models only
exist for `noiselevel-Q`), then tests for a linear noise-level trend using the SAME
[2, 1, 0, -1, -2] (cleanest-to-noisiest) contrast weights already used for the univariate
SNR-trend in `group_level_all_ROI.ipynb`/`univariate_group-level.ipynb` -- positive trend =
stronger model-fit when the signal is cleaner.

Statistics reuse the bootstrap functions already defined above (`bootstrap_one_sample`,
`bootstrap_group_difference`) rather than introducing classical t-tests, to stay internally
consistent with the rest of this notebook (unlike the univariate SNR-trend, which used
`ttest_1samp`/`ttest_ind` -- this notebook's own convention throughout has been bootstrap-only).

**Only computed if all 5 noise levels are available.** A trend contrast over a partial subset
of noise levels wouldn't use the same, correctly-centered weights as the full 5-level contrast,
so it's skipped entirely (not attempted with reweighted partial contrasts) until every tag has
been processed.

In [ ]:
COMMON_MODELS = ['syllable', 'speaker']
SNR_TREND_WEIGHTS = dict(zip(NOISE_LEVEL_TAGS, [2, 1, 0, -1, -2]))

if len(results_by_tag) == 0:
    print('No noise levels available yet -- nothing to combine.')
    model_fit_by_noise_df = pd.DataFrame(columns=['participant_id', 'group', 'ROI', 'model', 'fit', 'noise_level'])
else:
    model_fit_by_noise_df = pd.concat([
        result['model_fit_df'][result['model_fit_df'].model.isin(COMMON_MODELS)].assign(noise_level=tag)
        for tag, result in results_by_tag.items()
    ], ignore_index=True)
model_fit_by_noise_df['noise_level'] = pd.Categorical(
    model_fit_by_noise_df['noise_level'], categories=NOISE_LEVEL_TAGS, ordered=True)

print(f'{len(model_fit_by_noise_df)} rows across {model_fit_by_noise_df.noise_level.nunique()} noise level(s)')


In [ ]:
NOISE_LEVEL_PALETTE = dict(zip(NOISE_LEVEL_TAGS, sns.color_palette('crest', len(NOISE_LEVEL_TAGS))))


def plot_model_fit_by_noise_level(model_fit_by_noise_df, model_name, roi_order):
    """Combined view: x=ROI, hue=noise_level (sequential 'crest' palette, cleanest=lightest to
    noisiest=darkest) -- mirrors univariate_fmri/group_level_all_ROI.ipynb's
    plot_roi_beta_by_snr, adapted for RSA model-fit values instead of GLM betas. One figure per
    (model, group) -- pooling both groups on one hue dimension already used for noise level would
    need a second (group) facet or hue, which gets crowded with 5 noise levels already in play.
    """
    figs = {}
    for group_name in GROUP_ORDER:
        group_df = model_fit_by_noise_df[
            (model_fit_by_noise_df.model == model_name) & (model_fit_by_noise_df.group == group_name)
        ]
        fig, ax = plt.subplots(1, 1, figsize=(0.5 * len(roi_order) + 2, 4), dpi=300)
        sns.boxplot(data=group_df, x='ROI', y='fit', hue='noise_level', order=roi_order,
                   hue_order=NOISE_LEVEL_TAGS, palette=NOISE_LEVEL_PALETTE,
                   fliersize=0, linewidth=1, ax=ax)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
        sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1), title='Noise level')
        ax.axhline(y=0, color='0.5', linestyle='--', linewidth=0.5)
        ax.set_ylabel('Model-fit (RDM correlation)')
        ax.set_title(f'model-{model_name}, group-{group_name}, across noise levels')
        fig.tight_layout()
        sns.despine(ax=ax)
        figs[group_name] = fig
    return figs


for model_name in COMMON_MODELS:
    figs = plot_model_fit_by_noise_level(model_fit_by_noise_df, model_name, CORTICAL_ROI_LIST)
    for group_name, fig in figs.items():
        fig.savefig(os.path.join(out_dir, f'boxplot_model-{model_name}_group-{group_name}_by-noiselevel.png'))
        fig.savefig(os.path.join(out_dir, f'boxplot_model-{model_name}_group-{group_name}_by-noiselevel.svg'))
        plt.close(fig)

print('Cross-noise-level boxplots saved for models:', COMMON_MODELS)


#### Linear noise-level trend

Per-subject, per-ROI, per-model trend score: the model-fit values across all 5 noise levels,
dotted with `[2, 1, 0, -1, -2]` (same convention as the univariate SNR-trend). Requires complete
5-level coverage per (subject, ROI, model) -- any incomplete combination is dropped (should not
normally happen, since `run_rsa_for_noise_level` already requires >=3 subjects per group before
returning a result, but a subject present in some noise levels and missing from others would
still be possible if RSA was only run for a subset of that subject's sessions).

In [ ]:
if len(results_by_tag) < len(NOISE_LEVEL_TAGS):
    print(f'Only {len(results_by_tag)}/{len(NOISE_LEVEL_TAGS)} noise levels available '
         f'({list(results_by_tag.keys())}) -- skipping the linear noise-level trend '
         '(needs all 5 levels for a correctly-centered contrast).')
    fit_trend_df = None
else:
    fit_wide = model_fit_by_noise_df.pivot_table(
        index=['participant_id', 'group', 'ROI', 'model'],
        columns='noise_level', values='fit', observed=True,
    )[NOISE_LEVEL_TAGS]
    complete_mask = fit_wide.notna().all(axis=1)
    n_dropped = int((~complete_mask).sum())
    if n_dropped > 0:
        print(f'Dropping {n_dropped} (participant, ROI, model) combination(s) with incomplete noise-level coverage.')
    fit_wide = fit_wide[complete_mask]

    trend_weights = [SNR_TREND_WEIGHTS[tag] for tag in NOISE_LEVEL_TAGS]
    fit_trend_df = fit_wide.dot(trend_weights).rename('trend').reset_index()
    print(f'{len(fit_trend_df)} (participant, ROI, model) trend scores computed')


In [ ]:
if fit_trend_df is not None:
    within_group_trend_rows = []
    for roi in CORTICAL_ROI_LIST:
        for model_name in COMMON_MODELS:
            for group_name in ['CWNS', 'CWS']:
                trend_values = fit_trend_df[
                    (fit_trend_df.ROI == roi) & (fit_trend_df.model == model_name) & (fit_trend_df.group == group_name)
                ]['trend']
                if len(trend_values) == 0:
                    continue
                result = bootstrap_one_sample(trend_values)
                within_group_trend_rows.append({'ROI': roi, 'model': model_name, 'group': group_name, **result})
    within_group_trend_df = pd.DataFrame(within_group_trend_rows)
    within_group_trend_df['p_fdr'] = multipletests(within_group_trend_df['p_value'], method='fdr_bh')[1]
    within_group_trend_df.to_csv(os.path.join(out_dir, 'within_group_trend_stats.csv'), index=False)

    between_group_trend_rows = []
    for roi in CORTICAL_ROI_LIST:
        for model_name in COMMON_MODELS:
            trend_cws = fit_trend_df[
                (fit_trend_df.ROI == roi) & (fit_trend_df.model == model_name) & (fit_trend_df.group == 'CWS')
            ]['trend']
            trend_cwns = fit_trend_df[
                (fit_trend_df.ROI == roi) & (fit_trend_df.model == model_name) & (fit_trend_df.group == 'CWNS')
            ]['trend']
            if len(trend_cws) == 0 or len(trend_cwns) == 0:
                continue
            result = bootstrap_group_difference(trend_cws, trend_cwns)
            between_group_trend_rows.append({'ROI': roi, 'model': model_name, **result})
    between_group_trend_df = pd.DataFrame(between_group_trend_rows)
    between_group_trend_df['p_fdr'] = multipletests(between_group_trend_df['p_value'], method='fdr_bh')[1]
    between_group_trend_df.to_csv(os.path.join(out_dir, 'between_group_trend_stats.csv'), index=False)

    print('Within-group FDR-significant trend tests (p_fdr < 0.05):')
    print(within_group_trend_df[within_group_trend_df.p_fdr < 0.05].sort_values('p_fdr'))
    print()
    print('Between-group FDR-significant trend tests (p_fdr < 0.05):')
    print(between_group_trend_df[between_group_trend_df.p_fdr < 0.05].sort_values('p_fdr'))


In [ ]:
def plot_trend_box_strip(fit_trend_df, within_group_trend_df, model_name, roi_order):
    """Box+strip of per-subject linear noise-level trend scores, x=ROI, hue=group -- mirrors
    group_level_all_ROI.ipynb's plot_roi_trend_box_strip (univariate SNR-trend), adapted for
    RSA model-fit trend scores instead of GLM beta trend scores.
    """
    trend_model_df = fit_trend_df[fit_trend_df.model == model_name]

    fig, ax = plt.subplots(1, 1, figsize=(0.5 * len(roi_order) + 2, 4), dpi=300)
    sns.stripplot(data=trend_model_df, x='ROI', y='trend', hue='group', order=roi_order,
                 hue_order=GROUP_ORDER, palette=GROUP_PALETTE,
                 dodge=True, linewidth=0.5, size=3, legend=None, ax=ax, zorder=2)
    sns.boxplot(data=trend_model_df, x='ROI', y='trend', hue='group', order=roi_order,
               hue_order=GROUP_ORDER, palette=GROUP_PALETTE,
               dodge=True, linewidth=1, fliersize=0, ax=ax, zorder=1)
    _outline_only(ax)

    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1), title='Group')
    ax.axhline(y=0, color='0.5', linestyle='--', linewidth=0.5)
    ax.set_ylabel('Linear noise-level trend (model-fit)')
    ax.set_title(f'model-{model_name}: linear noise-level trend')

    y_top = trend_model_df['trend'].max()
    asterisk_y = _add_asterisk_headroom(ax, y_top)
    sig_rois = within_group_trend_df[
        (within_group_trend_df.model == model_name) & (within_group_trend_df.p_fdr < 0.05)
    ]['ROI'].unique()
    for roi in sig_rois:
        if roi in roi_order:
            ax.text(roi_order.index(roi), asterisk_y, '*', ha='center', va='bottom',
                    fontsize=14, fontweight='bold')

    fig.tight_layout()
    sns.despine(ax=ax)
    return fig


if fit_trend_df is not None:
    for model_name in COMMON_MODELS:
        fig = plot_trend_box_strip(fit_trend_df, within_group_trend_df, model_name, CORTICAL_ROI_LIST)
        fig.savefig(os.path.join(out_dir, f'boxplot_trend_model-{model_name}.png'))
        fig.savefig(os.path.join(out_dir, f'boxplot_trend_model-{model_name}.svg'))
        plt.close(fig)
    print('Noise-level trend boxplots saved for models:', COMMON_MODELS)


### Summary

In [ ]:
for tag, result in results_by_tag.items():
    group_comparison_df = result['group_comparison_df']
    within_group_significance_df = result['within_group_significance_df']
    print(f'=== noiselevel-{tag} ===')
    print(f'  {len(result["sub_list_cwns"])} CWNS, {len(result["sub_list_cws"])} CWS, '
         f'{len(result["model_rdms"])} models: {list(result["model_rdms"].keys())}')
    print(f'  FDR-significant between-group tests (p_fdr < 0.05): {(group_comparison_df.p_fdr < 0.05).sum()} / {len(group_comparison_df)}')
    print(f'  FDR-significant within-group tests (p_fdr < 0.05): {(within_group_significance_df.p_fdr < 0.05).sum()} / {len(within_group_significance_df)}')
    print()

print(f'Noise levels processed: {len(results_by_tag)}/{len(NOISE_LEVEL_TAGS)} ({list(results_by_tag.keys())})')
if fit_trend_df is not None:
    print(f'Linear noise-level trend: {len(within_group_trend_df)} within-group tests, '
         f'{len(between_group_trend_df)} between-group tests (models: {COMMON_MODELS})')
else:
    print('Linear noise-level trend: skipped (not all 5 noise levels available yet)')
print(f'\nAll outputs saved to {out_dir}')
